# Swap-pair labeling on Colab

Labels `pair_slates.jsonl` with a GPU-hosted teacher, producing the same
`pair_labels.json` the local Ollama script produces — same schema, same prompt,
same id-set validation — so results from the two are directly comparable.

**Runtime → Change runtime type → GPU** before running. T4 (free) works for 8B/14B
in 4-bit; A100 (Pro) is much faster and needed for 32B.

## Why re-label everything, not just the remainder

Mixing two teachers in one corpus makes the learning curve uninterpretable — you
can't tell whether a change came from more data or a different judge. Labeling all
15,025 pairs gives you both teachers on identical pairs, which turns
"is a better teacher worth it?" into a measurement.


## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA:", torch.cuda.is_available())

## 2. Install vLLM

vLLM is used rather than plain `transformers` for two reasons: continuous batching
(the whole point of moving to a GPU), and **guided JSON decoding**, which constrains
output to the schema the same way Ollama's `format` did locally. Without that
constraint a 4-bit model drops or invents fields often enough to matter.

Takes ~3-5 minutes.

In [ ]:
%pip install -q vllm
# Colab often needs a runtime restart after this. If the next cell errors on import,
# use Runtime -> Restart session, then continue from cell 3 (skip this one).

## 2b. CUDA 13 preload (required after the vLLM install)

vLLM 0.26 is built against CUDA 13; Colab's torch is cu128. The cu13 runtime is present
but not on the loader path. This cell fixes that and verifies the import — **run it
after restarting the session**, before anything that touches vLLM.

In [ ]:
# vLLM 0.26 ships a CUDA 13 build while Colab's torch is cu128. The cu13 runtime IS
# installed (as a vllm dependency) but is not on the dynamic loader's search path, so
# `import vllm` dies with "libcudart.so.13: cannot open shared object file".
#
# Setting LD_LIBRARY_PATH here does NOT fix it - glibc caches the search path when the
# process starts, so a mid-session change is ignored. Preloading with RTLD_GLOBAL does:
# once the library is resident under that soname, vLLM's own dlopen() finds it.
#
# Driver 580+ is required (it supports both the CUDA 12 and 13 runtimes side by side).
# Check with nvidia-smi in cell 2 - on an older driver this will load but fail later
# with a CUDA error, and the fix is pinning vllm to a cu12-era build instead.
import ctypes, glob, os
CU13 = '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib'
if os.path.isdir(CU13):
    os.environ['LD_LIBRARY_PATH'] = CU13 + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    ok = 0
    for so in sorted(glob.glob(f'{CU13}/*.so*')):
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL); ok += 1
        except OSError:
            pass  # dependency-order failures are normal; only the vllm import matters
    print(f"preloaded {ok} cu13 libs")
else:
    print("no cu13 dir - preload not needed")

import torch, vllm
print("torch", torch.__version__, "| vllm", vllm.__version__, "| CUDA", torch.cuda.is_available())
print("gpu  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## 3. Mount Drive and upload the slates

Put `pair_slates.jsonl` (17.9 MB, from `npm run pairs:slates`) somewhere in your
Drive. Checkpoints are written back to Drive so a Colab disconnect — free sessions
cap around 12h and idle-timeout sooner — never costs more than one batch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/smartswaps'   # <- adjust if you put it elsewhere
import os; os.makedirs(DRIVE_DIR, exist_ok=True)
print(os.listdir(DRIVE_DIR))

## 4. Configuration

Same model as the local Ollama run (`qwen3:14b`), so this is a straight port of
step 3 onto a GPU — not a change of teacher. `MAX_PER_CALL` is held at 15 to match
the local run exactly.

One difference worth knowing: Ollama serves a GGUF-quantized qwen3:14b, while this
uses bitsandbytes 4-bit. Same weights, slightly different quantization. On a task
whose output is four coarse 0-3 ordinals that should be immaterial, but it is the
reason to re-label the whole corpus here rather than merging these labels into the
partial local run — one teacher per corpus keeps the learning curve interpretable.

In [ ]:
MODEL        = "Qwen/Qwen3-14B"
QUANT        = "bitsandbytes"   # set to None on an A100 with spare VRAM - full precision is faster
MAX_PER_CALL = 15               # matches scripts/label_swap_pairs_ollama.py
TAG          = "qwen3-14b-colab"

# T4 (free tier) fits 14B in 4-bit. A100 (Pro) runs it comfortably unquantized.

## 5. Load the slates

In [ ]:
import json, os
SLATES = os.path.join(DRIVE_DIR, 'pair_slates.jsonl')
slates = [json.loads(l) for l in open(SLATES) if l.strip()]
total_pairs = sum(len(s['candidates']) for s in slates)
print(f"{len(slates)} slates, {total_pairs} pairs")

## 6. Prompt and schema

Byte-identical to `scripts/label_swap_pairs_ollama.py`. Do not edit — any change
here breaks comparability with the local run.

Note `compact_food` drops the `sensory` block on purpose: those step-2 vectors came
back badly compressed (`sour`/`bitter` were exactly 1 for ~90% of foods), and
anchoring the teacher on a bad number is worse than letting it use the food name.

In [ ]:
SYSTEM_INSTRUCTION = """
You are rating food swaps for a nutrition app. The user bought the SOURCE food.
The app wants to suggest a healthier CANDIDATE they would actually accept in its
place. Rate every candidate in the list.

For EVERY candidate, copy its "pair_id" into your output EXACTLY as given. This
is how results are matched back, so it must not be altered, dropped, or invented.
Output one object per candidate - no more, no fewer.

Rate three INDEPENDENT axes, each an integer 0-3.

taste_fit - would this plausibly replace the source in a real meal?
  Judge by what the foods ARE, from their names and culinary role. Think about
  flavour, texture, and how the food is eaten - not nutrient numbers.
  3 = a direct substitute; you could swap it into the same meal unnoticed
      (fruit yoghurt -> skyr; white bread -> wholegrain bread)
  2 = same role, noticeably different eating experience
      (cow milk -> oat milk; beef mince -> turkey mince)
  1 = same broad occasion but not really a substitute
      (crisps -> almonds)
  0 = not a substitute at all; nobody would accept this swap
      (yoghurt -> garlic; cola -> beef broth)
  Be strict here. Most cross-category pairs are 0 even when the nutrition is
  better. A swap nobody would eat is worthless however healthy it is.

nutrition_gain - is the candidate meaningfully better nutritionally?
  Use the per-100g values given. Consider the direction that matters for THIS
  kind of food (added sugar for sweet foods and drinks, saturated fat for fats
  and meats, fibre for grains, salt for savoury foods, protein where relevant).
  3 = clearly and substantially better
  2 = moderately better
  1 = marginally better, or better in one way and worse in another
  0 = no real gain, or a net loss

effect_fit - does it do the same JOB in the day, and behave better in the body?
  Use culinary_role, prep_state, and the effect block (glycemic_load, satiety,
  caffeine, alcohol, time_of_day).
  3 = same role and time of day, and lower glycemic load or higher satiety
  2 = same role and time of day, similar physiological effect
  1 = role or timing partly mismatched
  0 = wrong role entirely (a cooking ingredient for a ready-to-eat food, a snack
      for a main, a drink for a solid), or it adds caffeine or alcohol

verdict - the overall call on whether the app should ever show this swap:
  "good"     = show it confidently. Requires taste_fit >= 2 AND nutrition_gain >= 2.
  "marginal" = defensible but not compelling; show only if nothing better exists.
  "bad"      = never show it. ANY candidate with taste_fit 0 is "bad", no matter
               how good its nutrition is.

Judge each candidate on its own merits. Do not assume the list is ordered
best-to-worst - it is not, and it usually contains some clearly bad candidates.
Do not feel obliged to spread ratings across the range: if every candidate is
bad, rate them all bad.

The German name (name_de) is authoritative when the English name reads like an
awkward database translation.
"""

print(len(SYSTEM_INSTRUCTION), "chars")

In [ ]:
AXES = ("taste_fit", "nutrition_gain", "effect_fit")
VERDICTS = ("good", "marginal", "bad")

BATCH_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "pair_id": {"type": "string"},
            **{ax: {"type": "integer", "minimum": 0, "maximum": 3} for ax in AXES},
            "verdict": {"type": "string", "enum": list(VERDICTS)},
        },
        "required": ["pair_id", *AXES, "verdict"],
    },
}

def compact_food(view):
    return {"name": view["name"], "name_de": view["name_de"],
            "category": view["swiss_category"], "per100g": view["per100g"],
            "culinary_role": view["culinary_role"], "prep_state": view["prep_state"],
            "effect": view["effect"]}

def build_prompt(slate, candidates):
    source = compact_food(slate["source"])
    items = [{"pair_id": c["pair_id"], **compact_food(c["candidate"])} for c in candidates]
    return (f"SOURCE FOOD (what the user bought):\n{json.dumps(source, ensure_ascii=False)}\n\n"
            f"Rate these {len(items)} candidate swaps:\n{json.dumps(items, ensure_ascii=False)}")

def parse_label(raw):
    if not isinstance(raw, dict): raise ValueError("not an object")
    pid = raw.get("pair_id")
    if not isinstance(pid, str) or not pid: raise ValueError(f"bad pair_id {pid!r}")
    out = {"pair_id": pid}
    for ax in AXES:
        v = raw.get(ax)
        if isinstance(v, bool) or not isinstance(v, int) or not 0 <= v <= 3:
            raise ValueError(f"{pid}: {ax}={v!r}")
        out[ax] = v
    if raw.get("verdict") not in VERDICTS: raise ValueError(f"{pid}: verdict={raw.get('verdict')!r}")
    out["verdict"] = raw["verdict"]
    return out

## 7. Load the model

In [ ]:
from vllm import LLM, SamplingParams
import vllm.sampling_params as _sp

# vLLM renamed guided decoding to "structured outputs" in the V1 engine, so the class
# to use depends on the version Colab happens to install that week:
#   <= ~0.10  SamplingParams(guided_decoding=GuidedDecodingParams(json=...))
#   >= ~0.11  SamplingParams(structured_outputs=StructuredOutputsParams(json=...))
# Detect rather than pin - this is the one part of the pipeline that must not silently
# degrade. Without schema enforcement a 4-bit model emits malformed JSON often enough
# that the id-set check would start rejecting whole batches.
def make_sampling_params(schema, **kw):
    if hasattr(_sp, "StructuredOutputsParams"):
        return SamplingParams(structured_outputs=_sp.StructuredOutputsParams(json=schema), **kw)
    if hasattr(_sp, "GuidedDecodingParams"):
        return SamplingParams(guided_decoding=_sp.GuidedDecodingParams(json=schema), **kw)
    raise RuntimeError(
        "No structured-output class found in vllm.sampling_params. Available: "
        + str([n for n in dir(_sp) if n[:1].isupper()])
    )

print("structured output via:",
      "StructuredOutputsParams" if hasattr(_sp, "StructuredOutputsParams") else "GuidedDecodingParams")

kwargs = dict(model=MODEL, max_model_len=8192, gpu_memory_utilization=0.90, dtype="auto")
if QUANT:
    kwargs["quantization"] = QUANT

llm = LLM(**kwargs)
tok = llm.get_tokenizer()
print("loaded", MODEL)

## 8. Label

Builds every request up front and hands the whole list to vLLM, which batches them
continuously — this is where the speedup over sequential Ollama calls comes from.

The id-set check is the same one the local script uses, and matters *more* here:
guided decoding guarantees the JSON *shape* but not that the model echoed the right
`pair_id`s, so anything mismatched is dropped and retried rather than trusted.

In [ ]:
import time, math

OUT = os.path.join(DRIVE_DIR, f'pair_labels_{TAG}.json')
results = {}
if os.path.exists(OUT):
    results = {r['pair_id']: r for r in json.load(open(OUT))}
    print(f"resuming with {len(results)} existing labels")

# chunk every slate into <= MAX_PER_CALL requests, skipping already-labeled pairs
requests = []
for s in slates:
    pending = [c for c in s['candidates'] if c['pair_id'] not in results]
    for i in range(0, len(pending), MAX_PER_CALL):
        chunk = pending[i:i+MAX_PER_CALL]
        requests.append((build_prompt(s, chunk), {c['pair_id'] for c in chunk}))
print(f"{len(requests)} requests to run")

sp = make_sampling_params(BATCH_SCHEMA, temperature=0.2, max_tokens=2048)

def run(batch):
    prompts = [tok.apply_chat_template(
        [{"role":"system","content":SYSTEM_INSTRUCTION},{"role":"user","content":p}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False) for p,_ in batch]
    return llm.generate(prompts, sp)

CHUNK = 200          # checkpoint to Drive this often
t0, ok, bad = time.time(), 0, 0
for start in range(0, len(requests), CHUNK):
    batch = requests[start:start+CHUNK]
    outs = run(batch)
    for (_, expected), o in zip(batch, outs):
        try:
            parsed = [parse_label(x) for x in json.loads(o.outputs[0].text)]
            got = [p['pair_id'] for p in parsed]
            if len(got) != len(set(got)) or set(got) != expected:
                bad += 1; continue
            for p in parsed: results[p['pair_id']] = p
            ok += 1
        except Exception:
            bad += 1
    json.dump(list(results.values()), open(OUT,'w'), indent=1)
    el = time.time()-t0
    frac = (start+len(batch))/len(requests)
    print(f"{start+len(batch)}/{len(requests)} requests | {len(results)}/{total_pairs} pairs "
          f"| ok {ok} bad {bad} | {el/60:.1f}m elapsed, ETA {el/max(frac,1e-9)/60*(1-frac):.1f}m")

print(f"\nDone. {len(results)}/{total_pairs} labeled -> {OUT}")
print(f"validation failures: {bad} of {ok+bad} requests")

## 9. Download

Then locally:

```bash
cp ~/Downloads/pair_labels_<tag>.json scripts/pair_labels.json
npm run eval:curve
```

Keep the Ollama run's output as `pair_labels_qwen3-14b-ollama.json` rather than
overwriting it — with both files you can run the curve against each and see whether
the teacher was actually the bottleneck.

In [ ]:
from google.colab import files
files.download(OUT)